### Data Cleaning
The aim of this project is to create an xPass model to identify the best passers in football. The model will be trained using event data from every game across the top 5 leagues in the 25/26 season. To begin, this data must be cleaned, so all metrics needed to train the model are available. The model will be built in stages, using Opta's framework for xPass, training both a logisitic regressor and gradient booster.

Model 1 - Geomertic passing information:
- (x,y) pass start and end locations
- pass distance
- pass angle

Model 2 - Contextual passing information:
- was the pass played with foot or head?
- was it a cross?
- was it open play or a set piece?
- was it a ground pass or airborne?

Model 3 - Contextual information about the possession:
- how many passes in the possession so far?
- how far was the ball carried before the pass?
- how fast has the ball moved vertically up the pitch in the previous three actions?
- was the previous action by a teammate or an opponent?
- what was the distance and angle of the previous pass?

In [17]:
# -------------------------------------------
# 1. Load the event data for the top 5 Leagues
# -------------------------------------------
from pathlib import Path
import pandas as pd
import numpy as np

leagues = ["bundesliga", 
           "la-liga",
           "ligue1",
           "prem",
           "serie-a"]
league_dfs = []

for league in leagues:
    league_events = pd.read_csv(Path.cwd().parent /f"data/raw/{league}-event-data-25-26.csv")

    league_events["league"] = league

    league_dfs.append(league_events)

# Concat the data into one df
events = pd.concat(league_dfs, ignore_index = True)

# Preserve original event order
events["_original_order"] = np.arange(len(events))

In [19]:
# -------------------------------------------
# 2. Filter for passes
# -------------------------------------------
passes_df = events[events["type"] == "Pass"]

In [21]:
# -------------------------------------------
# 3. Identify unique qualifiers
# -------------------------------------------
import ast
from collections import Counter

qualifier_counts = Counter()

for qualifiers in passes_df["qualifiers"].dropna():

    if isinstance(qualifiers, str):
        try:
            qualifiers = ast.literal_eval(qualifiers)
        except (ValueError, SyntaxError):
            continue

    if not isinstance(qualifiers, list):
        continue

    for qualifier in qualifiers:
        if not isinstance(qualifier, dict):
            continue

        qualifier_type = qualifier.get("type", {})
        qualifier_name = qualifier_type.get("displayName")

        if qualifier_name:
            qualifier_counts[qualifier_name] += 1

qualifier_summary = pd.DataFrame(
    qualifier_counts.items(),
    columns=["qualifier", "count"]
).sort_values("count", ascending=False).reset_index(drop=True)

display(qualifier_summary)

,qualifier,count
0,Zone,1689005
1,PassEndY,1689005
2,PassEndX,1689005
3,Length,1689005
4,Angle,1689005
5,Longball,218743
6,Chipped,149746
7,HeadPass,88485
8,ThrowIn,63820
9,Cross,61639


In [23]:
# -------------------------------------------
# 4. Extract relevant qualifiers
# -------------------------------------------
def extract_pass_features(qualifiers):

    features = {"pass_length" : None,
                "pass_angle" : None,
                "is_chipped" : 0,
                "is_head_pass" : 0,
                "is_cross" : 0,
                "is_freekick" : 0,
                "is_goal_kick" : 0,
                "is_corner" : 0,
                "is_indirect_fk" : 0}

    if isinstance(qualifiers, str):
        try: 
            qualifiers = ast.literal_eval(qualifiers)
        except (ValueError, SyntaxError):
            return features

    if not isinstance(qualifiers, list):
        return features

    for q in qualifiers:

        if not isinstance(q, dict):
            continue

        name = q.get("type", {}).get("displayName")
        value = q.get("value")

        if name == "Length":
            features["pass_length"] = value

        elif name == "Angle":
            features["pass_angle"] = value

        elif name == "Chipped":
            features["is_chipped"] = 1

        elif name == "HeadPass":
            features["is_head_pass"] = 1

        elif name == "Cross":
            features["is_cross"] = 1

        elif name == "FreekickTaken":
            features["is_freekick"] = 1

        elif name == "GoalKick":
            features["is_goal_kick"] = 1

        elif name == "CornerTaken":
            features["is_corner"] = 1

        elif name == "IndirectFreekickTaken":
            features["is_indirect_fk"] = 1

    return features

feature_data = passes_df["qualifiers"].apply(extract_pass_features)

feature_df = pd.DataFrame(
    feature_data.tolist(),
    index=passes_df.index
)

passes_df = pd.concat([passes_df, feature_df], axis=1)

In [25]:
# -------------------------------------------
# 5. Define set pieces
# -------------------------------------------

set_piece_features = [
    "is_freekick",
    "is_goal_kick",
    "is_corner",
    "is_indirect_fk"
]

passes_df["is_set_piece"] = (
    passes_df[set_piece_features].max(axis=1)
)

passes_df["is_open_play"] = (
    1 - passes_df["is_set_piece"]
)

In [27]:
# -------------------------------------------
# 6. Define an algorithm to assign possessions
# -------------------------------------------
def assign_possessions(df):

    df = df.copy()

    control_events = {
        "Pass",
        "BallRecovery",
        "Interception",
        "Clearance",
        "BallTouch",
        "TakeOn",
        "Claim",
        "BlockedPass",
        "KeeperPickup",
        "PenaltyFaced",
        "CrossNotClaimed",
        "Smother",
        "GoodSkill",
        "ChanceMissed"
    }

    possession_ids = []

    possession_id = 0
    possession_team = None

    for _, event in df.iterrows():

        event_type = event["type"]
        team = event["team"]
        outcome = event["outcome_type"]

        # Aerials are contested events, not confirmed possession
        if event_type == "Aerial":
            possession_ids.append(pd.NA)
            continue

        # Fouls
        if event_type == "Foul":
        
            if outcome == "Successful":
                possession_id += 1
                possession_team = team
                possession_ids.append(possession_id)

            else:
                possession_ids.append(pd.NA)

            continue
        
        # Offside stoppages
        if event_type == "OffsideGiven":
            possession_ids.append(pd.NA)
            possession_team = None
            continue

        if event_type in {"OffsidePass", "OffsideProvoked"}:
            possession_ids.append(pd.NA)
            continue

        # Corner awards: only successful awards establish possession
        if event_type == "CornerAwarded":

            if outcome == "Successful":
                possession_id += 1
                possession_team = team
                possession_ids.append(possession_id)
            else:
                possession_ids.append(pd.NA)

            continue

        # Control events
        if event_type in control_events:

            if possession_team != team:
                possession_id += 1
                possession_team = team

            possession_ids.append(possession_id)
            continue

        # Tackles
        if event_type == "Tackle":
            possession_ids.append(pd.NA)
        
            if outcome == "Successful":
                possession_team = None
        
            continue

        # Goals
        if event_type == "Goal":

            if possession_team != team:
                possession_id += 1
                possession_team = team
        
            possession_ids.append(possession_id)
            possession_team = None
        
            continue

        if event_type in {"Challenge", 
                          "Save", 
                          "SubstitutionOn", 
                          "SubstitutionOff", 
                          "End",
                          "Dispossessed",
                          "Card",
                          "KeeperSweeper",
                          "Punch",
                          "FormationSet",
                          "FormationChange",
                          "ShieldBallOpp",
                          "Error"}:
            possession_ids.append(pd.NA)
            continue

        # Shots
        if event_type in {"MissedShots", "SavedShot","ShotOnPost"}:
        
            if possession_team != team:
                possession_id += 1
                possession_team = team
        
            possession_ids.append(possession_id)
            continue

        if event_type == "Start":
            possession_ids.append(pd.NA)
            possession_team = None
            continue

        # Other events inherit the current possession where possible
        if possession_team is not None:
            possession_ids.append(possession_id)
        else:
            possession_ids.append(pd.NA)

    df["possession_id"] = pd.array(possession_ids, dtype="Int64")

    return df

events = assign_possessions(events)

In [28]:
# -------------------------------------------
# 7. Apply the algorithm to the data
# -------------------------------------------
events_with_possessions = events.copy()
events_with_possessions["possession_id"] = pd.NA

for match_id in events_with_possessions["game_id"].unique():

    mask = events_with_possessions["game_id"].eq(match_id)

    df = (
        events_with_possessions.loc[mask]
        .sort_values(
            ["period", "minute", "second", "_original_order"],
            na_position="last"
        )
        .copy()
    )

    assigned = assign_possessions(df)

    events_with_possessions.loc[
        assigned.index, "possession_id"
    ] = assigned["possession_id"]

In [33]:
# -------------------------------------------
# 8. Calculate the passes in the possession
# -------------------------------------------
# Transfer possession IDs to the pass DataFrame
passes_df["possession_id"] = (
    passes_df.index.map(events_with_possessions["possession_id"])
)

passes_df = passes_df.sort_index().copy()

passes_df["passes_in_possession"] = (
    passes_df.groupby(["game_id", "possession_id"]).cumcount()
)

passes_df.loc[
    passes_df["game_id"].eq(1901061),
    [
        "minute", "second", "team", "player",
        "possession_id", "passes_in_possession"
    ]
].head(20)

,minute,second,team,player,possession_id,passes_in_possession
2106520,0,0.0,Atalanta,Gianluca Scamacca,1,0
2106521,0,1.0,Atalanta,Mario Pasalic,1,1
2106522,0,3.0,Atalanta,Marten de Roon,1,2
2106523,0,5.0,Atalanta,Isak Hien,1,3
2106524,0,8.0,Atalanta,Giorgio Scalvini,1,4
2106525,0,12.0,Atalanta,Charles De Ketelaere,1,5
2106526,0,15.0,Atalanta,Isak Hien,1,6
2106527,0,18.0,Atalanta,Mario Pasalic,1,7
2106528,0,21.0,Atalanta,Berat Djimsiti,1,8
2106529,0,23.0,Pisa,Antonio Caracciolo,2,0


In [35]:
# -------------------------------------------
# 9. Calculate ball carrying distance
# -------------------------------------------

# Start with a clean copy of the pass data
passes_df = passes_df.drop(
    columns=[
        "previous_event_index",
        "previous_end_x",
        "previous_end_y",
        "previous_possession_id",
        "previous_team",
        "previous_type",
        "carry_distance",
        "pre_pass_ball_movement"
    ],
    errors="ignore"
).copy()


# 1. Sort all events into chronological order within each match
ordered_events = events_with_possessions.sort_values(
    ["game_id", "period", "minute", "second", "_original_order"],
    na_position="last"
).copy()


# 2. Find the immediately preceding event in each match
event_indices = pd.Series(
    ordered_events.index,
    index=ordered_events.index
)

ordered_events["previous_event_index"] = (
    event_indices.groupby(
        ordered_events["game_id"], sort=False
    ).shift(1)
)


# 3. Map the preceding event index onto each pass
passes_df["previous_event_index"] = (
    passes_df.index.to_series().map(
        ordered_events["previous_event_index"]
    )
)


# 4. Retrieve information about each preceding event
previous_events = ordered_events[
    ["end_x", "end_y", "possession_id", "team", "type"]
].rename(columns={
    "end_x": "previous_end_x",
    "end_y": "previous_end_y",
    "possession_id": "previous_possession_id",
    "team": "previous_team",
    "type": "previous_type"
})

for column in previous_events.columns:
    passes_df[column] = passes_df["previous_event_index"].map(
        previous_events[column]
    )


# 5. Identify passes where the preceding event belongs
# to the same possession
current_possession = pd.to_numeric(
    passes_df["possession_id"], errors="coerce"
).to_numpy(dtype=float, na_value=np.nan)

previous_possession = pd.to_numeric(
    passes_df["previous_possession_id"], errors="coerce"
).to_numpy(dtype=float, na_value=np.nan)

same_possession = pd.Series(
    np.isfinite(current_possession)
    & np.isfinite(previous_possession)
    & (current_possession == previous_possession),
    index=passes_df.index
)


# 6. Calculate pre-pass ball movement
passes_df["pre_pass_ball_movement"] = np.nan

valid_coordinates = (
    pd.to_numeric(passes_df["x"], errors="coerce").notna()
    & pd.to_numeric(passes_df["y"], errors="coerce").notna()
    & pd.to_numeric(passes_df["previous_end_x"], errors="coerce").notna()
    & pd.to_numeric(passes_df["previous_end_y"], errors="coerce").notna()
)

valid_movement = same_possession & valid_coordinates

passes_df.loc[valid_movement, "pre_pass_ball_movement"] = np.hypot(
    passes_df.loc[valid_movement, "x"]
    - passes_df.loc[valid_movement, "previous_end_x"],
    passes_df.loc[valid_movement, "y"]
    - passes_df.loc[valid_movement, "previous_end_y"]
)

passes_df["has_pre_pass_movement"] = (
    passes_df["pre_pass_ball_movement"].notna().astype(int)
)

In [37]:
# -------------------------------------------
# 10. Define if the previous action was the same team
# -------------------------------------------

passes_df["previous_team"] = passes_df["previous_event_index"].map(
    events_with_possessions["team"]
)

passes_df["previous_action_same_team"] = np.where(
    passes_df["previous_team"].isna(),
    np.nan,
    (passes_df["team"] == passes_df["previous_team"]).astype(int)
)

display(
    passes_df["previous_action_same_team"].value_counts(
        dropna=False
    )
)

previous_action_same_team
1.0    1379473
0.0     309532
Name: count, dtype: int64

In [39]:
# -------------------------------------------
# 11. Define previous pass angle and length
# -------------------------------------------
previous_pass_features = passes_df[
    ["pass_length", "pass_angle"]
].copy()

passes_df["previous_pass_length"] = (
    passes_df["previous_event_index"].map(
        previous_pass_features["pass_length"]
    )
)

passes_df["previous_pass_angle"] = (
    passes_df["previous_event_index"].map(
        previous_pass_features["pass_angle"]
    )
)

passes_df["previous_pass_length"] = pd.to_numeric(
    passes_df["previous_pass_length"], errors="coerce"
)

passes_df["previous_pass_angle"] = pd.to_numeric(
    passes_df["previous_pass_angle"], errors="coerce"
)

In [43]:
# -------------------------------------------
# 12. Convert passing angles
# -------------------------------------------

# Convert angles to numeric
passes_df["pass_angle"] = pd.to_numeric(
    passes_df["pass_angle"], errors="coerce"
)

passes_df["previous_pass_angle"] = pd.to_numeric(
    passes_df["previous_pass_angle"], errors="coerce"
)

# Calculate sine and cosine
passes_df["pass_angle_sin"] = np.sin(passes_df["pass_angle"])
passes_df["pass_angle_cos"] = np.cos(passes_df["pass_angle"])

passes_df["previous_pass_angle_sin"] = np.sin(
    passes_df["previous_pass_angle"]
)

passes_df["previous_pass_angle_cos"] = np.cos(
    passes_df["previous_pass_angle"]
)

In [45]:
# -------------------------------------------
# 13. Calculate vertical ball progression
# -------------------------------------------

# Create a clean chronological copy
ordered_events = events_with_possessions.sort_values(
    ["game_id", "period", "minute", "second", "_original_order"],
    na_position="last"
).copy()

# Retrieve the three immediately preceding event indices
for i in range(1, 4):
    ordered_events[f"previous_event_index_{i}"] = (
        ordered_events.groupby("game_id", sort=False)
        .apply(lambda group: pd.Series(
            group.index, index=group.index
        ).shift(i))
        .reset_index(level=0, drop=True)
    )

event_details = events_with_possessions[
    ["possession_id", "period", "end_x", "minute", "second"]
]

for i in range(1, 4):
    previous_indices = ordered_events[
        f"previous_event_index_{i}"
    ]

    for column in event_details.columns:
        ordered_events[f"previous_{column}_{i}"] = (
            previous_indices.map(event_details[column])
        )

for column in [
    "previous_event_index_1",
    "previous_event_index_2",
    "previous_event_index_3"
]:
    passes_df[column] = passes_df.index.to_series().map(
        ordered_events[column]
    )

for column in [
    "previous_possession_id_1",
    "previous_possession_id_2",
    "previous_possession_id_3",
    "previous_period_1",
    "previous_period_2",
    "previous_period_3",
    "previous_end_x_1",
    "previous_end_x_3",
    "previous_minute_1",
    "previous_minute_3",
    "previous_second_1",
    "previous_second_3"
]:
    passes_df[column] = passes_df.index.to_series().map(
        ordered_events[column]
    )

current_possession = pd.to_numeric(
    passes_df["possession_id"], errors="coerce"
).to_numpy(dtype=float, na_value=np.nan)

valid_history = passes_df["possession_id"].notna().to_numpy()

for i in range(1, 4):
    previous_possession = pd.to_numeric(
        passes_df[f"previous_possession_id_{i}"],
        errors="coerce"
    ).to_numpy(dtype=float, na_value=np.nan)

    valid_history &= (
        np.isfinite(previous_possession)
        & (previous_possession == current_possession)
    )

    valid_history &= (
        passes_df[f"previous_period_{i}"]
        .eq(passes_df["period"])
        .fillna(False)
        .to_numpy()
    )

valid_history &= (
    passes_df["previous_end_x_3"].notna().to_numpy()
    & passes_df["previous_end_x_1"].notna().to_numpy()
)

time_earliest = (
    passes_df["previous_minute_3"] * 60
    + passes_df["previous_second_3"]
)

time_latest = (
    passes_df["previous_minute_1"] * 60
    + passes_df["previous_second_1"]
)

elapsed_seconds = time_latest - time_earliest

valid_rate = valid_history & elapsed_seconds.gt(0).to_numpy()

passes_df["vertical_ball_progression_rate"] = np.nan

passes_df.loc[valid_rate, "vertical_ball_progression_rate"] = (
    (
        passes_df.loc[valid_rate, "previous_end_x_1"]
        - passes_df.loc[valid_rate, "previous_end_x_3"]
    )
    / elapsed_seconds.loc[valid_rate]
)

print(passes_df["vertical_ball_progression_rate"].describe())

print(
    "Valid values:",
    passes_df["vertical_ball_progression_rate"].notna().sum()
)

print(
    "Missing values:",
    passes_df["vertical_ball_progression_rate"].isna().sum()
)

/var/folders/9w/g5l2j37s6_vd_bmq9vg4qlv40000gp/T/ipykernel_3124/3785732581.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda group: pd.Series(
/var/folders/9w/g5l2j37s6_vd_bmq9vg4qlv40000gp/T/ipykernel_3124/3785732581.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda group: pd.Series(
/var/folders/9w/g5l2j37s6_vd_bmq9vg4qlv40000gp/T/ipykernel_3124/3785732581.py:15: FutureWarning: Dat

count    721127.000000
mean          0.467989
std           3.469934
min         -51.600000
25%          -1.277778
50%           0.455556
75%           2.337500
max          65.400000
Name: vertical_ball_progression_rate, dtype: float64
Valid values: 721127
Missing values: 967878


In [56]:
# -------------------------------------------
# 14. Create the final df
# -------------------------------------------

xpass_df = passes_df[
    [
        "x", "y",
        "end_x", "end_y",
        "pass_length", 
        "pass_angle_sin",
        "pass_angle_cos",
        "is_head_pass",
        "is_cross",
        "is_set_piece",
        "is_chipped",
        "passes_in_possession",
        "pre_pass_ball_movement",
        "vertical_ball_progression_rate",
        "previous_action_same_team",
        "previous_pass_length",
        "previous_pass_angle_sin",
        "previous_pass_angle_cos",
        "outcome_type",
        "league",
        "season",
        "game_id",
        "player_id",
        "player",
        "minute",
        "second"
    ]
].copy()

xpass_df["outcome"] = (
    xpass_df["outcome_type"] == "Successful"
).astype(int)

xpass_df = xpass_df.drop(columns="outcome_type")

xpass_df.to_csv(Path.cwd().parent / "data/cleaned/xPass_features.csv")